In [1]:
import ssl
ssl._create_default_https_context = ssl._create_unverified_context
import kagglehub
import pandas as pd
import os
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer, AutoModelForSeq2SeqLM, T5Tokenizer, T5ForConditionalGeneration

import OpenAttack as oa
import torch.nn.functional as F
from datasets import Dataset
import numpy as np
from tqdm import tqdm

c:\Users\Priyanka\AppData\Local\Programs\Python\Python38\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
path = kagglehub.dataset_download("babykrishnaml/unified-mental-health-nlp-dataset")

files = os.listdir(path)
csv_path = [f for f in files if f.endswith(".csv")][0]

df = pd.read_csv(os.path.join(path, csv_path))
df.head()

,text,label_emotion,label_topic,label_distortion,response,dataset_source
0,"Now if he does off himself, everyone will thin...",['neutral'],NaN,NaN,NaN,goemotions_split
1,WHY THE FUCK IS BAYLESS ISOING,['anger'],NaN,NaN,NaN,goemotions_split
2,To make her feel threatened,['fear'],NaN,NaN,NaN,goemotions_split
3,Dirty Southern Wankers,['annoyance'],NaN,NaN,NaN,goemotions_split
4,OmG pEyToN iSn'T gOoD eNoUgH tO hElP uS iN tHe...,['surprise'],NaN,NaN,NaN,goemotions_split


In [3]:
# Standardize
df.columns = [c.lower() for c in df.columns]

TEXT_COL = "text"
LABEL_COL = "label_emotion"

df = df.dropna(subset=[TEXT_COL, LABEL_COL])
# Binary mapping
def map_label(x):
    if x.lower() == "['neutral']":
        return 0
    else:
        return 1

df["label"] = df[LABEL_COL].apply(map_label)

df = df[[TEXT_COL, "label"]]
df.dropna(inplace=True)

In [4]:
dataset = Dataset.from_pandas(df)
dataset = dataset.train_test_split(test_size=0.2)

train_ds = dataset["train"]
test_ds = dataset["test"]

formatted_data = [
    {"x": test_ds["text"][i], "y": test_ds["label"][i]}
    for i in range(min(50, len(test_ds["text"])))
]

In [5]:
def load_model(path):
    # detect model type from path
    if "Roberta" in path:
        tokenizer = AutoTokenizer.from_pretrained("roberta-base")
    else:
        tokenizer = AutoTokenizer.from_pretrained(
            path,
            local_files_only=True
        )

    model = AutoModelForSequenceClassification.from_pretrained(
        path,
        low_cpu_mem_usage=True,
        local_files_only=True
    )

    return model, tokenizer

In [6]:
class TransformerClassifier(oa.Classifier):
    def __init__(self, model, tokenizer, device="cpu"):
        self.model = model
        self.tokenizer = tokenizer
        self.device = device

    def get_prob(self, input_):
        inputs = self.tokenizer(
            input_,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=128
        ).to(self.device)

        with torch.no_grad():
            outputs = self.model(**inputs)
        return outputs.logits.softmax(dim=1).cpu().numpy()

    def get_pred(self, input_):
        return self.get_prob(input_).argmax(axis=1)

In [11]:
attackers = {
    "textFooler (SEA)": oa.attackers.TextFoolerAttacker(),
    "SCPN": oa.attackers.SCPNAttacker(),
    "GAN": oa.attackers.GANAttacker()
}

c:\Users\Priyanka\AppData\Local\Programs\Python\Python38\lib\site-packages\nltk\corpus\reader\wordnet.py:1172: UserWarning: The multilingual functions are not available with this Wordnet version
  warnings.warn(


In [12]:
def run_attackeval(model, tokenizer, dataset, attacker, num_samples=50):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model.to(device)
    model.eval()

    clf = TransformerClassifier(model, tokenizer, device=device)
    attack_eval = oa.AttackEval(attacker,clf)
    results = attack_eval.eval(dataset, visualize=True)

In [7]:
trained_model = {}

trained_model["bert"] = (*load_model("./imbalanced/Bert"), test_ds)
trained_model["electra"] = (*load_model("./imbalanced/Electra"), test_ds)
trained_model["roberta"] = (*load_model("./imbalanced/Roberta"), test_ds)

In [14]:
all_results = {}

for name, (model, tokenizer, test_data) in trained_model.items():
    print(f"\n========== {name.upper()} ==========\n")

    for name, attacker in attackers.items():
        print(f"\n{name.upper()}")

        results = run_attackeval(
            model,
            tokenizer,
            formatted_data,
            attacker,
            num_samples=50 
        )


    all_results[name] = results


========== BERT ==========


TEXTFOOLER (SEA)
Sample: 1 =====================================================================
Label: 1 (99.99%) --> 0 (74.96%)            |                                   
                                            |                                   
My neighr got  a new        riding          |                                   
my neighr pose a unexampled disembarrass    | Running Time:            1.2353   
                                            | Query Exceeded:          no       
mower . I ' m still stuck with my push      | Victim Model Queries:    115      
mower . i ' m still stuck with my push      | Succeed:                 yes      
                                            |                                   
mower .                                     |                                   
lawn  .                                     |                                   
                                            |                  

In [ ]:
# new model

In [8]:
device = "cuda" if torch.cuda.is_available() else "cpu"
t5_tokenizer = T5Tokenizer.from_pretrained("t5-small")
t5_model = T5ForConditionalGeneration.from_pretrained("t5-small").to(device)
t5_model.eval()

T5ForConditionalGeneration(
  (shared): Embedding(32128, 512)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 512)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=512, out_features=512, bias=False)
              (k): Linear(in_features=512, out_features=512, bias=False)
              (v): Linear(in_features=512, out_features=512, bias=False)
              (o): Linear(in_features=512, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 8)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=512, out_features=2048, bias=False)
              (wo): Linear(in_features=2048, out_features=512, bias=False)
              (dropout): Drop

In [9]:
def predict(model, tokenizer, texts):
    inputs = tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=256,
        return_tensors="pt"
    )
    
    inputs = {k: v.to(device) for k, v in inputs.items()}
    
    with torch.no_grad():
        outputs = model(**inputs)
        probs = torch.softmax(outputs.logits, dim=-1)
    
    return probs.cpu().numpy(), np.argmax(probs.cpu().numpy(), axis=1)

In [10]:
def generate_context(sentence, num_return=5):
    all_results = []
    prompts = [
        f"rewrite the sentence by restructuring it while keeping meaning and sentiment: {sentence}",
        f"rephrase with different syntax but same meaning and sentiment: {sentence}",
        f"rewrite using a different grammatical structure but keeping the sentiment same: {sentence}",
    ]

    for p in prompts:
        inputs = t5_tokenizer(p, return_tensors="pt", truncation=True).to(device)

        outputs = t5_model.generate(
            **inputs,
            max_length=128,
            num_return_sequences = max(1, num_return // len(prompts)),
            do_sample=True,
            top_k=50,
            top_p=0.95
        )

        results = [
            t5_tokenizer.decode(o, skip_special_tokens=True)
            for o in outputs
        ]

        all_results.extend(results)

    return all_results

In [11]:
def best_attack(sentence, label, model, tokenizer, beam_width=5, steps=5):
    
    beam = [(sentence, 0)]  # (text, score)

    for _ in range(steps):
        new_beam = []

        for sent, _ in beam:
            candidates = generate_context(sent, num_return=5)

            for c in candidates:
                probs, _ = predict(model, tokenizer, [c])

                # improved scoring
                true_conf = probs[0][label]
                wrong_conf = np.max([p for i, p in enumerate(probs[0]) if i != label])
                
                score = wrong_conf - true_conf

                new_beam.append((c, score))

                # early success
                if np.argmax(probs[0]) != label and wrong_conf > 0.5:
                    return c

        # keep top-k
        beam = sorted(new_beam, key=lambda x: x[1], reverse=True)[:beam_width]

    return beam[0][0]

In [12]:
def run_attack(model, tokenizer, samples, max_samples=50):
    success = 0
    total = 0
    
    for sample in tqdm(samples[:max_samples]):
        x = sample["x"]
        y = sample["y"]
        
        probs_orig, pred_orig = predict(model, tokenizer, [x])
        
        # skip incorrect originals
        if pred_orig[0] != y:
            continue
        
        # generate adversarial
        x_adv = best_attack(x, y, model, tokenizer)
        
        probs_adv, pred_adv = predict(model, tokenizer, [x_adv])
        
        if pred_adv[0] != y:
            success += 1
        
        total += 1
    
    return success / total if total > 0 else 0

FINAL

In [13]:
models = {}
tokenizers = {}

for name, (model, tokenizer, test_data) in trained_model.items():    
    model.to(device)
    model.eval()
    
    models[name] = model
    tokenizers[name] = tokenizer


for name in models:
    print(f"\nAttacking {name.upper()}...")
    
    asr = run_attack(
        models[name],
        tokenizers[name],
        formatted_data,
        max_samples=50
    )
    
    print(f"Attack Success Rate (ASR): {asr:.4f}")


Attacking BERT...


100%|██████████| 50/50 [07:48<00:00,  9.36s/it]


Attack Success Rate (ASR): 0.9545

Attacking ELECTRA...


100%|██████████| 50/50 [04:32<00:00,  5.45s/it]


Attack Success Rate (ASR): 0.9762

Attacking ROBERTA...


100%|██████████| 50/50 [06:19<00:00,  7.59s/it]

Attack Success Rate (ASR): 0.9767


1 para

In [20]:
models = {}
tokenizers = {}

for name, (model, tokenizer, test_data) in trained_model.items():    
    model.to(device)
    model.eval()
    
    models[name] = model
    tokenizers[name] = tokenizer


for name in models:
    print(f"\nAttacking {name.upper()}...")
    
    asr = run_attack(
        models[name],
        tokenizers[name],
        formatted_data,
        max_samples=50
    )
    
    print(f"Attack Success Rate (ASR): {asr:.4f}")


Attacking BERT...


100%|██████████| 50/50 [02:31<00:00,  3.03s/it]


Attack Success Rate (ASR): 0.7826

Attacking ELECTRA...


100%|██████████| 50/50 [02:46<00:00,  3.32s/it]


Attack Success Rate (ASR): 0.6522

Attacking ROBERTA...


100%|██████████| 50/50 [03:10<00:00,  3.81s/it]

Attack Success Rate (ASR): 0.8444


3 para

In [24]:
models = {}
tokenizers = {}

for name, (model, tokenizer, test_data) in trained_model.items():    
    model.to(device)
    model.eval()
    
    models[name] = model
    tokenizers[name] = tokenizer


for name in models:
    print(f"\nAttacking {name.upper()}...")
    
    asr = run_attack(
        models[name],
        tokenizers[name],
        formatted_data,
        max_samples=50
    )
    
    print(f"Attack Success Rate (ASR): {asr:.4f}")


Attacking BERT...


100%|██████████| 50/50 [02:47<00:00,  3.35s/it]


Attack Success Rate (ASR): 0.7826

Attacking ELECTRA...


100%|██████████| 50/50 [02:56<00:00,  3.53s/it]


Attack Success Rate (ASR): 0.7826

Attacking ROBERTA...


100%|██████████| 50/50 [02:24<00:00,  2.88s/it]

Attack Success Rate (ASR): 0.7556


5 para

In [28]:
models = {}
tokenizers = {}

for name, (model, tokenizer, test_data) in trained_model.items():    
    model.to(device)
    model.eval()
    
    models[name] = model
    tokenizers[name] = tokenizer


for name in models:
    print(f"\nAttacking {name.upper()}...")
    
    asr = run_attack(
        models[name],
        tokenizers[name],
        formatted_data,
        max_samples=50
    )
    
    print(f"Attack Success Rate (ASR): {asr:.4f}")


Attacking BERT...


100%|██████████| 50/50 [02:33<00:00,  3.08s/it]


Attack Success Rate (ASR): 0.6739

Attacking ELECTRA...


100%|██████████| 50/50 [02:30<00:00,  3.02s/it]


Attack Success Rate (ASR): 0.7826

Attacking ROBERTA...


100%|██████████| 50/50 [02:28<00:00,  2.98s/it]

Attack Success Rate (ASR): 0.7111
